# PyOpt: LoRA fine-tuning Qwen3-4B to optimize Python functions

Training and evaluation notebook. See the [README](README.md) for the results table and the limitations.

**About the saved outputs.** The outputs below come from the original run on Kaggle (2× Tesla T4, September 2026) with the **v1.1 split** (`data/v1.1/`). In that split, 13 of the 40 test prompts are renamed copies of training prompts, so the in-distribution numbers in section 6 are affected by train/test leakage. The dataset generator now builds a leak-free **v1.2 split** (`data/`), which has not been trained on yet.

The notebook was cleaned after the run (template residue, a duplicate model load, Kaggle-only paths and warning spam removed, paths made configurable). It was **not** re-executed after the cleanup, because that needs a GPU. Exact package versions of the run are in `requirements-train.txt`.

To rerun: set `PYOPT_DATA_DIR` (default `data`, the v1.2 split) and `PYOPT_OUT_DIR` (default `results/v1.2`), then run all cells on a CUDA GPU. Re-score any output file offline with `python evaluate.py`.


In [15]:
!pip install -q unsloth

In [ ]:
import json
import os
from pathlib import Path

# The recorded run read the v1.1 split from a Kaggle dataset and wrote to /kaggle/working.
DATA_DIR = Path(os.environ.get("PYOPT_DATA_DIR", "data"))
OUT_DIR = Path(os.environ.get("PYOPT_OUT_DIR", "results/v1.2"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA_DIR, "| outputs:", OUT_DIR)


In [16]:
from unsloth import FastLanguageModel
import torch

print("Unsloth loaded")
print("CUDA available:", torch.cuda.is_available())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth loaded
CUDA available: True


## 1. Load Qwen3-4B-Instruct-2507 (4-bit) and add LoRA adapters

LoRA r=32, alpha=32 on all attention and MLP projections. The chat template is set to `qwen3-instruct` after loading.


In [17]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

==((====))==  Unsloth 2026.9.11: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 3.257 GiB
no_split classes   : ['Qwen3DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 11.166 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.95 GiB  weights  1.760 GiB  free 11.192 GiB  reserve 11.166 GiB
  cuda:1  budget  13.00 GiB  weights  1.497 GiB  free 11.499 GiB  reserve 11.009 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.391 -> 12.952 GiB, cuda:1 14.440 -> 12.996 GiB (memory th

In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.9.11 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [29]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-instruct",
)

## 2. Load the dataset


In [32]:
from datasets import load_dataset

files = {
    "train": str(DATA_DIR / "pyopt_train.jsonl"),
    "validation": str(DATA_DIR / "pyopt_val.jsonl"),
    "test": str(DATA_DIR / "pyopt_test.jsonl"),
}

pyopt = load_dataset("json", data_files=files)

dataset = pyopt["train"]
val_dataset = pyopt["validation"]
test_dataset = pyopt["test"]

print("Train:", len(dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 160
Validation: 30
Test: 40


Inspect one training example:


In [33]:
dataset[0]

{'id': 'string_concat-002',
 'category': 'string_concat',
 'fn': 'merge_words_2',
 'original': 'def merge_words_2(chunks):\n    out = ""\n    for piece in chunks:\n        out += piece\n    return out\n',
 'optimized': 'def merge_words_2(chunks):\n    return "".join(chunks)\n',
 'constraints': 'chunks is a list of strings.',
 'conversations': [{'role': 'user',
   'content': 'Optimize this Python function while preserving its behavior.\n\nInput constraints:\n- chunks is a list of strings.\n\nReturn only the optimized Python function.\n\n```python\ndef merge_words_2(chunks):\n    out = ""\n    for piece in chunks:\n        out += piece\n    return out\n```'},
  {'role': 'assistant',
   'content': '```python\ndef merge_words_2(chunks):\n    return "".join(chunks)\n```'}]}

## 3. Base model: generate on the test split

Greedy decoding, at most 300 new tokens. Raw outputs go to `OUT_DIR/base_outputs.jsonl`.


In [34]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

example = test_dataset[0]
prompt = example["conversations"][0]["content"]

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=False,
)

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("CATEGORY:", example["category"])
print("\nPROMPT:\n", prompt)
print("\nQWEN ANSWER:\n", answer)
print("\nEXPECTED:\n", example["optimized"])

CATEGORY: no_change

PROMPT:
 Optimize this Python function while preserving its behavior.

Input constraints:
- values is a list of integers. The function is already efficient and concise for this task; do not rewrite it unnecessarily.

Return only the optimized Python function.

```python
def map_squares_3(values):
    return [entry * entry for entry in values]
```

QWEN ANSWER:
 ```python
def map_squares_3(values):
    return [v ** 2 for v in values]
```

EXPECTED:
 def map_squares_3(values):
    return [entry * entry for entry in values]



In [36]:
import json
from pathlib import Path

baseline_results = []

for i, example in enumerate(test_dataset):
    prompt = example["conversations"][0]["content"]

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    baseline_results.append({
        "id": example["id"],
        "category": example["category"],
        "original": example["original"],
        "expected": example["optimized"],
        "model_output": answer,
    })

    print(f"{i+1}/40  {example['category']}  {example['id']}")

with (OUT_DIR / "base_outputs.jsonl").open("w") as f:
    for row in baseline_results:
        f.write(json.dumps(row) + "\n")


1/40  no_change  no_change-003
2/40  no_change  no_change-028
3/40  list_concat  list_concat-003
4/40  string_concat  string_concat-021
5/40  list_concat  list_concat-019
6/40  repeated_sort  repeated_sort-022
7/40  membership  membership-026
8/40  repeated_sum  repeated_sum-024
9/40  repeated_sum  repeated_sum-013
10/40  repeated_sum  repeated_sum-031
11/40  membership  membership-008
12/40  string_concat  string_concat-037
13/40  list_concat  list_concat-000
14/40  list_concat  list_concat-034
15/40  repeated_sum  repeated_sum-034
16/40  list_concat  list_concat-015
17/40  repeated_sort  repeated_sort-025
18/40  repeated_sort  repeated_sort-007
19/40  repeated_sum  repeated_sum-008
20/40  repeated_sort  repeated_sort-004
21/40  list_concat  list_concat-016
22/40  repeated_sum  repeated_sum-029
23/40  membership  membership-025
24/40  no_change  no_change-023
25/40  no_change  no_change-009
26/40  list_concat  list_concat-025
27/40  membership  membership-013
28/40  repeated_sort  rep

## 4. LoRA fine-tuning

2 epochs (40 steps), effective batch 8, lr 2e-4, 8-bit AdamW. `train_on_responses_only` masks the prompt, so the loss is computed on the assistant answer only. The validation split is passed in but no eval loss was logged in this run.


In [38]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]

    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]

    return {"text": texts}


train_formatted = dataset.map(
    formatting_prompts_func,
    batched=True,
)

val_formatted = val_dataset.map(
    formatting_prompts_func,
    batched=True,
)

print(train_formatted[0]["text"])

<|im_start|>user
Optimize this Python function while preserving its behavior.

Input constraints:
- chunks is a list of strings.

Return only the optimized Python function.

```python
def merge_words_2(chunks):
    out = ""
    for piece in chunks:
        out += piece
    return out
```<|im_end|>
<|im_start|>assistant
```python
def merge_words_2(chunks):
    return "".join(chunks)
```<|im_end|>



In [39]:
FastLanguageModel.for_training(model);  # trailing ; hides the long model repr


In [40]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    args=SFTConfig(
        dataset_text_field="text",

        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        num_train_epochs=2,

        learning_rate=2e-4,
        warmup_ratio=0.1,

        logging_steps=1,

        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",

        seed=3407,
        report_to="none",

        output_dir=str(OUT_DIR / "checkpoints"),
    ),
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [41]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

Unsloth: Auto-detected instruction_part = '<|im_start|>user\n' and response_part = '<|im_start|>assistant\n'


In [42]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 160 | Num Epochs = 2 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Step,Training Loss
1,0.757743
2,0.701004
3,1.165122
4,0.294568
5,0.541523
6,0.442073
7,0.431101
8,0.250645
9,0.124377
10,0.048838


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/pyopt_checkpoints/checkpoint-40/tokenizer_config.json.


In [43]:
save_path = OUT_DIR / "pyopt_lora"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Saved PyOpt LoRA to:", save_path)

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/pyopt_lora/tokenizer_config.json.


Saved PyOpt LoRA to: /kaggle/working/pyopt_lora


## 5. Fine-tuned model: generate on the test split


In [44]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model);  # trailing ; hides the long model repr


In [45]:
import json
from pathlib import Path

pyopt_results = []

for i, example in enumerate(test_dataset):
    prompt = example["conversations"][0]["content"]

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    pyopt_results.append({
        "id": example["id"],
        "category": example["category"],
        "original": example["original"],
        "expected": example["optimized"],
        "model_output": answer,
    })

    print(f"{i+1}/40  {example['category']}  {example['id']}")

1/40  no_change  no_change-003
2/40  no_change  no_change-028
3/40  list_concat  list_concat-003
4/40  string_concat  string_concat-021
5/40  list_concat  list_concat-019
6/40  repeated_sort  repeated_sort-022
7/40  membership  membership-026
8/40  repeated_sum  repeated_sum-024
9/40  repeated_sum  repeated_sum-013
10/40  repeated_sum  repeated_sum-031
11/40  membership  membership-008
12/40  string_concat  string_concat-037
13/40  list_concat  list_concat-000
14/40  list_concat  list_concat-034
15/40  repeated_sum  repeated_sum-034
16/40  list_concat  list_concat-015
17/40  repeated_sort  repeated_sort-025
18/40  repeated_sort  repeated_sort-007
19/40  repeated_sum  repeated_sum-008
20/40  repeated_sort  repeated_sort-004
21/40  list_concat  list_concat-016
22/40  repeated_sum  repeated_sum-029
23/40  membership  membership-025
24/40  no_change  no_change-023
25/40  no_change  no_change-009
26/40  list_concat  list_concat-025
27/40  membership  membership-013
28/40  repeated_sort  rep

In [46]:
output_path = OUT_DIR / "finetuned_outputs.jsonl"

with output_path.open("w") as f:
    for row in pyopt_results:
        f.write(json.dumps(row) + "\n")

print("Saved:", output_path)
print("Total results:", len(pyopt_results))

Saved: /kaggle/working/pyopt_finetuned_outputs.jsonl
Total results: 40


In [47]:
for i in range(5):
    print("=" * 80)
    print("ID:", pyopt_results[i]["id"])
    print("\nBASE:")
    print(baseline_results[i]["model_output"])
    print("\nFINETUNED:")
    print(pyopt_results[i]["model_output"])
    print("\nEXPECTED:")
    print(pyopt_results[i]["expected"])

ID: no_change-003

BASE:
```python
def map_squares_3(values):
    return [v ** 2 for v in values]
```

FINETUNED:
```python
def map_squares_3(values):
    return [entry * entry for entry in values]
```

EXPECTED:
def map_squares_3(values):
    return [entry * entry for entry in values]

ID: no_change-028

BASE:
```python
def map_squares_28(xs):
    return [x ** 2 for x in xs]
```

FINETUNED:
```python
def map_squares_28(xs):
    return [x * x for x in xs]
```

EXPECTED:
def map_squares_28(xs):
    return [x * x for x in xs]

ID: list_concat-003

BASE:
```python
def squares_3(items):
    return [number * number for number in items]
```

FINETUNED:
```python
def squares_3(items):
    output = []
    for number in items:
        output.append(number * number)
    return output
```

EXPECTED:
def squares_3(items):
    output = []
    for number in items:
        output.append(number * number)
    return output

ID: string_concat-021

BASE:
```python
def merge_words_21(parts):
    return ""

## 6. Score: correctness and speedup against the original function

A candidate is correct if it returns the same value as the original on two inputs; the speedup is timed on the larger input with `timeit` (100 calls, argument copying included). "Meaningfully faster" means speedup > 1.05.

**Leakage caveat:** 13 of these 40 test prompts are renamed copies of training prompts. `evaluate.py --train data/v1.1/pyopt_train.jsonl` reports the scores separately for the 27 clean prompts.


In [50]:
import re

def extract_code(text):
    match = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

In [51]:
import timeit
import copy

def get_test_cases(category):
    if category == "membership":
        return [
            (list(range(100)), list(range(50, 150))),
            (list(range(1000)), list(range(500, 1500))),
        ]

    if category == "repeated_sum":
        return [
            (list(range(1, 101)),),
            (list(range(1, 1001)),),
        ]

    if category == "repeated_sort":
        return [
            (list(range(100, 0, -1)), 20),
            (list(range(1000, 0, -1)), 100),
        ]

    if category in {"list_concat", "no_change"}:
        return [
            (list(range(100)),),
            (list(range(1000)),),
        ]

    if category == "string_concat":
        return [
            (["abc"] * 100,),
            (["abc"] * 1000,),
        ]

    raise ValueError(category)

In [52]:
def evaluate_candidate(example, output_text):
    code = extract_code(output_text)

    try:
        namespace = {}
        exec(code, namespace)

        fn_name = example["fn"]

        if fn_name not in namespace:
            return {
                "valid": False,
                "correct": False,
                "speedup": None,
                "error": "Function name missing",
            }

        candidate = namespace[fn_name]

        original_ns = {}
        exec(example["original"], original_ns)
        original = original_ns[fn_name]

        cases = get_test_cases(example["category"])

        # Correctness
        for args in cases:
            expected = original(*copy.deepcopy(args))
            actual = candidate(*copy.deepcopy(args))

            if actual != expected:
                return {
                    "valid": True,
                    "correct": False,
                    "speedup": None,
                    "error": "Output mismatch",
                }

        # Benchmark on largest case
        args = cases[-1]

        original_time = timeit.timeit(
            lambda: original(*copy.deepcopy(args)),
            number=100,
        )

        candidate_time = timeit.timeit(
            lambda: candidate(*copy.deepcopy(args)),
            number=100,
        )

        return {
            "valid": True,
            "correct": True,
            "speedup": original_time / candidate_time,
            "error": None,
        }

    except Exception as e:
        return {
            "valid": False,
            "correct": False,
            "speedup": None,
            "error": str(e),
        }

In [53]:
comparison = []

for example, base, tuned in zip(
    test_dataset,
    baseline_results,
    pyopt_results,
):
    base_score = evaluate_candidate(
        example,
        base["model_output"],
    )

    tuned_score = evaluate_candidate(
        example,
        tuned["model_output"],
    )

    comparison.append({
        "id": example["id"],
        "category": example["category"],
        "base": base_score,
        "pyopt": tuned_score,
    })

print("Evaluated:", len(comparison))

Evaluated: 40


In [54]:
import statistics

def summarize(which):
    rows = [x[which] for x in comparison]

    valid = sum(x["valid"] for x in rows)
    correct = sum(x["correct"] for x in rows)

    speedups = [
        x["speedup"]
        for x in rows
        if x["correct"] and x["speedup"] is not None
    ]

    improved = sum(
        1 for x in speedups
        if x > 1.05
    )

    return {
        "valid_code": f"{valid}/40",
        "correct": f"{correct}/40",
        "meaningfully_faster": f"{improved}/{len(speedups)}",
        "median_speedup": statistics.median(speedups) if speedups else None,
    }

print("BASE QWEN")
print(summarize("base"))

print("\nPYOPT")
print(summarize("pyopt"))

BASE QWEN
{'valid_code': '40/40', 'correct': '40/40', 'meaningfully_faster': '29/40', 'median_speedup': 3.569178656723694}

PYOPT
{'valid_code': '40/40', 'correct': '40/40', 'meaningfully_faster': '36/40', 'median_speedup': 5.6533571951167545}


In [55]:
from collections import defaultdict
import statistics

categories = defaultdict(lambda: {
    "base_correct": 0,
    "pyopt_correct": 0,
    "base_speedups": [],
    "pyopt_speedups": [],
})

for row in comparison:
    cat = row["category"]

    if row["base"]["correct"]:
        categories[cat]["base_correct"] += 1
        categories[cat]["base_speedups"].append(
            row["base"]["speedup"]
        )

    if row["pyopt"]["correct"]:
        categories[cat]["pyopt_correct"] += 1
        categories[cat]["pyopt_speedups"].append(
            row["pyopt"]["speedup"]
        )

for cat, stats in categories.items():
    print("\n", cat)

    print(
        "Base median:",
        round(statistics.median(stats["base_speedups"]), 2)
    )

    print(
        "PyOpt median:",
        round(statistics.median(stats["pyopt_speedups"]), 2)
    )


 no_change
Base median: 0.98
PyOpt median: 1.0

 list_concat
Base median: 6.39
PyOpt median: 6.35

 string_concat
Base median: 1.26
PyOpt median: 1.27

 repeated_sort
Base median: 3.74
PyOpt median: 3.77

 membership
Base median: 1.01
PyOpt median: 11.58

 repeated_sum
Base median: 50.39
PyOpt median: 47.88


## 7. Out-of-distribution probe: 10 hand-written problems

Five new families (min, max, dict lookup, nested search, tuple) that are not in the training data. Only correctness is checked; the speedup field is not implemented for this set and is always `None`.


In [57]:
hard_tests = [
    {
        "id": "repeated_min_1",
        "category": "repeated_min",
        "fn": "shift_from_min",
        "original": """def shift_from_min(values):
    out = []
    for x in values:
        out.append(x - min(values))
    return out
""",
        "constraints": "values is a non-empty list of integers.",
    },
    {
        "id": "repeated_min_2",
        "category": "repeated_min",
        "fn": "distance_from_min",
        "original": """def distance_from_min(nums):
    result = []
    for n in nums:
        result.append(abs(n - min(nums)))
    return result
""",
        "constraints": "nums is a non-empty list of integers.",
    },
    {
        "id": "repeated_max_1",
        "category": "repeated_max",
        "fn": "normalize_by_max",
        "original": """def normalize_by_max(values):
    out = []
    for x in values:
        out.append(x / max(values))
    return out
""",
        "constraints": "values is a non-empty list of positive integers.",
    },
    {
        "id": "repeated_max_2",
        "category": "repeated_max",
        "fn": "gap_from_max",
        "original": """def gap_from_max(nums):
    result = []
    for x in nums:
        result.append(max(nums) - x)
    return result
""",
        "constraints": "nums is a non-empty list of integers.",
    },
    {
        "id": "repeated_dict_1",
        "category": "repeated_dict",
        "fn": "lookup_values",
        "original": """def lookup_values(keys, pairs):
    out = []
    for key in keys:
        if key in dict(pairs):
            out.append(dict(pairs)[key])
    return out
""",
        "constraints": "keys is a list of strings and pairs is a list of unique (string, integer) pairs.",
    },
    {
        "id": "repeated_dict_2",
        "category": "repeated_dict",
        "fn": "present_keys",
        "original": """def present_keys(keys, pairs):
    result = []
    for key in keys:
        if key in dict(pairs):
            result.append(key)
    return result
""",
        "constraints": "keys is a list of strings and pairs is a list of unique (string, integer) pairs.",
    },
    {
        "id": "nested_search_1",
        "category": "nested_search",
        "fn": "has_overlap",
        "original": """def has_overlap(a, b):
    for x in a:
        for y in b:
            if x == y:
                return True
    return False
""",
        "constraints": "a and b are lists of integers.",
    },
    {
        "id": "nested_search_2",
        "category": "nested_search",
        "fn": "common_exists",
        "original": """def common_exists(left, right):
    for x in left:
        for y in right:
            if x == y:
                return True
    return False
""",
        "constraints": "left and right are lists of integers.",
    },
    {
        "id": "repeated_tuple_1",
        "category": "repeated_tuple",
        "fn": "filter_allowed_tuple",
        "original": """def filter_allowed_tuple(items, allowed):
    out = []
    for x in items:
        if x in tuple(allowed):
            out.append(x)
    return out
""",
        "constraints": "items and allowed are lists of integers.",
    },
    {
        "id": "repeated_tuple_2",
        "category": "repeated_tuple",
        "fn": "keep_valid_tuple",
        "original": """def keep_valid_tuple(values, valid):
    result = []
    for x in values:
        if x in tuple(valid):
            result.append(x)
    return result
""",
        "constraints": "values and valid are lists of integers.",
    },
]

In [58]:
for example in hard_tests:
    example["prompt"] = f"""Optimize this Python function while preserving its behavior.

Input constraints:
- {example["constraints"]}

Return only the optimized Python function.

```python
{example["original"].strip()}
```"""

In [59]:
def generate_answer(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

In [60]:
hard_pyopt_results = []

for i, example in enumerate(hard_tests):
    answer = generate_answer(example["prompt"])

    hard_pyopt_results.append({
        **example,
        "model_output": answer,
    })

    print(f"PyOpt {i+1}/{len(hard_tests)}")

PyOpt 1/10
PyOpt 2/10
PyOpt 3/10
PyOpt 4/10
PyOpt 5/10
PyOpt 6/10
PyOpt 7/10
PyOpt 8/10
PyOpt 9/10
PyOpt 10/10


In [61]:
hard_base_results = []

with model.disable_adapter():
    for i, example in enumerate(hard_tests):
        answer = generate_answer(example["prompt"])

        hard_base_results.append({
            **example,
            "model_output": answer,
        })

        print(f"Base {i+1}/{len(hard_tests)}")

Base 1/10
Base 2/10
Base 3/10
Base 4/10
Base 5/10
Base 6/10
Base 7/10
Base 8/10
Base 9/10
Base 10/10


In [62]:
for i in range(len(hard_tests)):
    print("=" * 80)
    print(hard_tests[i]["id"])

    print("\nBASE:")
    print(hard_base_results[i]["model_output"])

    print("\nPYOPT:")
    print(hard_pyopt_results[i]["model_output"])

repeated_min_1

BASE:
```python
def shift_from_min(values):
    min_val = min(values)
    return [x - min_val for x in values]
```

PYOPT:
```python
def shift_from_min(values):
    min_value = min(values)
    out = []
    for x in values:
        out.append(x - min_value)
    return out
```
repeated_min_2

BASE:
```python
def distance_from_min(nums):
    min_val = min(nums)
    return [abs(n - min_val) for n in nums]
```

PYOPT:
```python
def distance_from_min(nums):
    min_value = min(nums)
    result = []
    for n in nums:
        result.append(abs(n - min_value))
    return result
```
repeated_max_1

BASE:
```python
def normalize_by_max(values):
    max_val = max(values)
    return [x / max_val for x in values]
```

PYOPT:
```python
def normalize_by_max(values):
    max_value = max(values)
    out = []
    for x in values:
        out.append(x / max_value)
    return out
```
repeated_max_2

BASE:
```python
def gap_from_max(nums):
    max_val = max(nums)
    return [max_val - x for

In [64]:
def hard_test_cases(example_id):
    cases = {
        "repeated_min_1": [
            ([3, 1, 5],),
            ([10],),
            ([-2, 4, 0],),
        ],
        "repeated_min_2": [
            ([3, 1, 5],),
            ([10],),
            ([-2, 4, 0],),
        ],
        "repeated_max_1": [
            ([1, 2, 4],),
            ([5],),
            ([3, 7, 2],),
        ],
        "repeated_max_2": [
            ([1, 2, 4],),
            ([5],),
            ([-3, 7, 2],),
        ],
        "repeated_dict_1": [
            (["a", "b"], [("a", 1), ("b", 2)]),
            (["b", "a"], [("a", 1), ("b", 2)]),
            (["a", "a"], [("a", 1), ("b", 2)]),
            (["x"], [("a", 1), ("b", 2)]),
        ],
        "repeated_dict_2": [
            (["a", "b"], [("a", 1), ("b", 2)]),
            (["b", "a"], [("a", 1), ("b", 2)]),
            (["a", "a"], [("a", 1), ("b", 2)]),
            (["x"], [("a", 1), ("b", 2)]),
        ],
        "nested_search_1": [
            ([1, 2], [3, 4]),
            ([1, 2], [2, 4]),
            ([1, 1], [3, 4]),
            ([], [1, 2]),
        ],
        "nested_search_2": [
            ([1, 2], [3, 4]),
            ([1, 2], [2, 4]),
            ([1, 1], [3, 4]),
            ([], [1, 2]),
        ],
        "repeated_tuple_1": [
            ([1, 2, 3], [2, 3]),
            ([1, 1, 2], [1]),
            ([], [1]),
            ([1, 2], []),
        ],
        "repeated_tuple_2": [
            ([1, 2, 3], [2, 3]),
            ([1, 1, 2], [1]),
            ([], [1]),
            ([1, 2], []),
        ],
    }

    return cases[example_id]

In [67]:
import copy
import timeit

def evaluate_hard(example, output_text):
    code = extract_code(output_text)

    try:
        original_ns = {}
        exec(example["original"], original_ns)
        original_fn = original_ns[example["fn"]]

        candidate_ns = {}
        exec(code, candidate_ns)

        if example["fn"] not in candidate_ns:
            return {
                "correct": False,
                "speedup": None,
                "error": "Function name missing"
            }

        candidate_fn = candidate_ns[example["fn"]]

        cases = hard_test_cases(example["id"])

        for args in cases:
            expected = original_fn(*copy.deepcopy(args))
            actual = candidate_fn(*copy.deepcopy(args))

            if actual != expected:
                return {
                    "correct": False,
                    "speedup": None,
                    "error": f"Mismatch on {args}: {actual!r} != {expected!r}"
                }

        # benchmark largest useful case separately
        return {
            "correct": True,
            "speedup": None,
            "error": None
        }

    except Exception as e:
        return {
            "correct": False,
            "speedup": None,
            "error": str(e)
        }

In [70]:
hard_comparison = []

for example, base, tuned in zip(
    hard_tests,
    hard_base_results,
    hard_pyopt_results,
):
    base_score = evaluate_hard(
        example,
        base["model_output"]
    )

    pyopt_score = evaluate_hard(
        example,
        tuned["model_output"]
    )

    hard_comparison.append({
        "id": example["id"],
        "base": base_score,
        "pyopt": pyopt_score,
    })

for row in hard_comparison:
    print(row["id"])
    print("  BASE :", row["base"])
    print("  PYOPT:", row["pyopt"])

repeated_min_1
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
repeated_min_2
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
repeated_max_1
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
repeated_max_2
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
repeated_dict_1
  BASE : {'correct': False, 'speedup': None, 'error': "Mismatch on (['b', 'a'], [('a', 1), ('b', 2)]): [1, 2] != [2, 1]"}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
repeated_dict_2
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
nested_search_1
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': False, 'speedup': None, 'error': 'Mismatc